In [1]:
import scanpy as sc
import anndata
from scipy import io
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np
import os
import pandas as pd

import scvelo as scv
import scanpy as sc
import cellrank as cr
import numpy as np
import pandas as pd
import anndata as ad

In [3]:
import session_info
session_info.show()

In [20]:
pip freeze

anndata==0.9.1
anyio==3.7.1
appdirs==1.4.4
appnope @ file:///opt/concourse/worker/volumes/live/6ca6f098-d773-4461-5c91-a24a17435bda/volume/appnope_1606859448531/work
argon2-cffi==21.3.0
argon2-cffi-bindings==21.2.0
arrow==1.2.3
asttokens @ file:///opt/conda/conda-bld/asttokens_1646925590279/work
async-lru==2.0.4
attrs==23.1.0
Babel==2.12.1
backcall @ file:///home/ktietz/src/ci/backcall_1611930011877/work
beautifulsoup4==4.12.2
bleach==6.0.0
blinker==1.7.0
bokeh==1.4.0
brotlipy==0.7.0
cellrank==2.0.0
certifi @ file:///home/conda/feedstock_root/build_artifacts/certifi_1707022139797/work/certifi
cffi @ file:///private/var/folders/sy/f16zz6x50xz3113nwtb9bvq00000gp/T/abs_1b0qzba5nr/croot/cffi_1670423213150/work
charset-normalizer @ file:///tmp/build/80754af9/charset-normalizer_1630003229654/work
click==8.1.3
cloudpickle==3.0.0
colorcet==3.0.1
contourpy @ file:///private/var/folders/sy/f16zz6x50xz3113nwtb9bvq00000gp/T/abs_17gskqgptz/croots/recipe/contourpy_1663827415320/work
cryptography @ f

In [ ]:
import os 
os.getcwd()

In [ ]:
os.chdir("yourpath")

# Seurat to Anndata 

In [ ]:
# load sparse matrix:
X = io.mmread("./integrated_seurat_rna_counts.mtx")

# create anndata object
adata = anndata.AnnData(
    X=X.transpose().tocsr()
)

# load cell metadata:
cell_meta = pd.read_csv("./integrated_seurat_metadata.csv")

# load gene names:
with open("./integrated_seurat_rna_counts.csv", 'r') as f:
    gene_names = f.read().splitlines()


In [ ]:
# set anndata observations and index obs by barcodes, var by gene names
adata.obs = cell_meta
adata.obs.index = adata.obs['barcode']
adata.var.index = gene_names

In [ ]:
# load dimensional reduction:
pca = pd.read_csv("./integrated_pca.csv")
pca.index = adata.obs.index


In [ ]:
# set pca and umap
adata.obsm['X_pca'] = pca.to_numpy()
adata.obsm['X_umap'] = np.vstack((adata.obs['UMAP_1'].to_numpy(), adata.obs['UMAP_2'].to_numpy())).T


In [ ]:
# plot a UMAP colored by sampleID to test:
sc.pl.umap(adata, color=['celltype'], frameon=False, save=True)


In [ ]:
adata.write('integrated_seurat_to_h5ad.h5ad')


# Load loom files 

In [ ]:
scv.settings.verbosity = 3
scv.settings.set_figure_params('scvelo', facecolor='white', dpi=100, frameon=False)
cr.settings.verbosity = 2

adata = sc.read_h5ad('integrated_seurat_to_h5ad.h5ad')

# load loom files for spliced/unspliced matrices for each sample:
d30_data = scv.read('../scvelo_30/gex_possorted_bam_96HZZ_d30.loom', cache=True)
d50_data = scv.read('../scvelo_d0/d50_gex_possorted_bam_8VM52.loom',cache=True)

In [ ]:
# rename barcodes in order to merge:
barcodes = [bc.split(':')[1] for bc in d30_data.obs.index.tolist()]
barcodes = ['D30_'+bc[0:len(bc)-1]+'-1'   for bc in barcodes]
barcodes
d30_data.obs.index = barcodes
d30_data.obs.index

# rename barcodes in order to merge:
barcodes = [bc.split(':')[1] for bc in d50_data.obs.index.tolist()]
barcodes = ['D50_'+bc[0:len(bc)-1]+'-1'   for bc in barcodes]
barcodes
d50_data.obs.index = barcodes
d50_data.obs.index

In [ ]:
# make variable names unique
d30_data.var_names_make_unique()
d50_data.var_names_make_unique()

# concatenate the three loom
ldata = d30_data.concatenate([d50_data])
ldata

scv.utils.clean_obs_names(adata)
scv.utils.clean_obs_names(ldata)

In [ ]:
# merge matrices into the original adata object
adata_merged = scv.utils.merge(adata, ldata)
adata_merged

sc.pl.umap(adata_merged, color=['celltype'], frameon=False, save=True)

adata_merged.write_h5ad("adata_integrated_seurat_and_loom_merged_ready_for_scvelo.h5ad")

# Normalizaation and pre-process before rna velocity computation

In [ ]:
os.chdir("../scvelo_integrated_seuratobj/")

adata_input = sc.read_h5ad("./adata_integrated_seurat_and_loom_merged_ready_for_scvelo.h5ad")

scv.pl.proportions(adata_input, groupby='celltype')

scv.pp.filter_and_normalize(adata_input, min_shared_counts=20, n_top_genes=5000)

scv.pp.log1p(adata_input)

sc.tl.pca(adata_input)

sc.pp.neighbors(adata_input, n_pcs=30, n_neighbors=30) #it kills kernel if it is not run on "scvelo" kernel!

scv.pp.moments(adata_input, n_pcs=30, n_neighbors=30)

# Compute RNA velocity -- stochastic

In [ ]:
scv.tl.velocity(adata_input, mode='stochastic')
scv.tl.velocity_graph(adata_input)

In [ ]:
adata_input.write_h5ad("adata_integrated_stochastic_velocity_computed_for_plotting.h5ad")

In [ ]:
# Plot velocity

In [ ]:
adata_input = sc.read_h5ad("./adata_integrated_stochastic_velocity_computed_for_plotting.h5ad")

In [43]:
adata_input.uns['celltype_colors']= ['#00C19A','#00A9FF','#FF61CC','#F8766D','#00BCD7','#00BE67','#E68613','#ABA300','#C77CFF','#8494FF','#0CB702','#d3d3d3']

In [ ]:
sc.pl.umap(adata_input, frameon=True,color='celltype', save= "UMAP_colors_added.pdf", legend_loc="on data",legend_fontsize = 'small')


In [ ]:
scv.pl.velocity_embedding_grid(adata_input, basis='umap', color='celltype', save='Integrated_stochastic_embedding_grid2_colors_same.pdf', title='Integrated Stochastic Velocity')
scv.pl.velocity_embedding_grid(adata_input, basis='umap', color='celltype', save='Integrated_stochastic_embedding_grid2_colors_same_scale_frameon.pdf', title='Integrated Stochastic Velocity',
                              scale=0.5,frameon=True)

scv.pl.velocity_embedding_stream(adata_input, frameon=True, basis='umap', color= 'celltype',save='Integrated_stochastic_embedding_stream_big_arrows_colors_same_frameon.pdf', title='Integrated Stochastic Velocity Stream')


In [ ]:
scv.pl.velocity_embedding_grid(adata_input, basis='umap', color='celltype', save='Integrated_stochastic_embedding_grid.pdf', title='Integrated Stochastic Velocity', scale=0.25)
# this section was giving "AttributeError: can't set attribute in python". downgradding pandas 2.0 to 1.3.5. worked! 
scv.pl.velocity_embedding_grid(adata_input, basis='umap', color='celltype', save='Integrated_stochastic_embedding_grid2.pdf', title='Integrated Stochastic Velocity')


In [ ]:
scv.pl.velocity_embedding_grid(adata_input, basis='umap', color='celltype', save='Integrated_stochastic_embedding_grid2.pdf', title='Integrated Stochastic Velocity',
                              scale=0.5)


In [ ]:
scv.pl.velocity_embedding_stream(adata_input, basis='umap', color= 'celltype',save='Integrated_stochastic_embedding_stream_big_arrows.pdf', title='Integrated Stochastic Velocity Stream')


In [ ]:
scv.logging.print_versions() 

In [ ]:
scv.tl.rank_velocity_genes(adata_input, groupby='celltype', min_corr=.3)
df = scv.DataFrame(adata_input.uns['rank_velocity_genes']['names'])
df.to_csv('Integrated_stochastic_important_TFnames_differential_velocity.csv')
df.head()

In [ ]:
adata_input.write_h5ad('./Integrated_stochastic_anndata.h5ad')

# Dynamical modelling --continue 

In [ ]:
dyn_input =  sc.read_h5ad("./adata_integrated_seurat_and_loom_merged_ready_for_scvelo.h5ad")

scv.settings.verbosity = 3  # show errors(0), warnings(1), info(2), hints(3)
scv.settings.presenter_view = True  # set max width size for presenter view
scv.settings.set_figure_params('scvelo')  # for beautified visualization

scv.pp.filter_and_normalize(dyn_input, min_shared_counts=20, n_top_genes=5000)
scv.pp.moments(dyn_input, n_pcs=30, n_neighbors=30)

scv.tl.recover_dynamics(dyn_input)
scv.tl.velocity(dyn_input, mode='dynamical')
scv.tl.velocity_graph(dyn_input)

dyn_input.write('./dynamicam_modelling_output_for_viz_integrated.h5ad', compression='gzip')

scv.pl.velocity_embedding_stream(dyn_input, basis='umap')
print(dyn_input.var.velocity_genes.sum())

top_genes = dyn_input.var['fit_likelihood'].sort_values(ascending=False).index[:300]
scv.pl.heatmap(dyn_input, var_names=top_genes, sortby='latent_time', col_color='celltype', n_convolve=100)

top_genes = dyn_input.var['fit_likelihood'].sort_values(ascending=False).index
scv.pl.scatter(dyn_input, basis=top_genes[:15], ncols=5, frameon=False,color='celltype')